In [1]:
# Importation des bibliothèques nécessaires
from pathlib import Path
import sys
import torch
from torch import nn
from collections import Counter

# Ajouter le dossier src au chemin Python pour pouvoir importer nos modules
project_src = Path.cwd().parent / "src"
if str(project_src) not in sys.path:
    sys.path.append(str(project_src))

# Importation de nos modules personnalisés
from pinkcc_ct_seg.data.dataset import BrainMRIDataset
from pinkcc_ct_seg.data.transforms import get_train_transforms, get_eval_transforms
from pinkcc_ct_seg.data.split import make_train_val_split
from pinkcc_ct_seg.data.loaders import make_loaders, make_weighted_sampler
from pinkcc_ct_seg.models.efficientnet_b0 import build_efficientnet_b0
from pinkcc_ct_seg.training.engine import train_one_epoch, validate_one_epoch
from pinkcc_ct_seg.training.utils import set_seed, get_device, save_checkpoint
from pinkcc_ct_seg.evaluation.metrics import compute_metrics

print("Imports OK ✅")

Imports OK ✅


In [2]:
# Configuration
SEED = 42
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 10
LR = 1e-3

DATA_DIR = Path("data/raw/brain_mri/Training")
TEST_DIR = Path("data/raw/brain_mri/Testing")

set_seed(SEED)
device = get_device()
print("Device:", device)
print("Train dir existe ?", DATA_DIR.exists())
print("Test dir existe ?", TEST_DIR.exists())

Device: cpu
Train dir existe ? True
Test dir existe ? True


In [3]:
# Définition des transformations pour les images
# - train_tfms : augmentation des données pour l'entraînement
# - eval_tfms  : juste redimensionnement pour validation/test
train_tfms = get_train_transforms(img_size=IMG_SIZE)
eval_tfms = get_eval_transforms(img_size=IMG_SIZE)

# Chargement de toutes les images d'entraînement
full_train_ds = BrainMRIDataset(DATA_DIR, transform=train_tfms)

# Extraction des labels (0 = pas de tumeur, 1 = tumeur)
labels = [full_train_ds.samples[i][1] for i in range(len(full_train_ds))]

print("Nb images total:", len(full_train_ds))
print("Distribution des classes:", Counter(labels))

# Séparation en train (80%) et validation (20%)
train_idx, val_idx = make_train_val_split(labels, val_size=0.2, random_state=SEED)
print("Train size:", len(train_idx))
print("Val size:", len(val_idx))

Nb images total: 2870
Distribution des classes: Counter({1: 2475, 0: 395})
Train size: 2296
Val size: 574


In [4]:
# Création des datasets pour chaque split
from torch.utils.data import Subset, DataLoader
full_eval_ds = BrainMRIDataset(DATA_DIR, transform=eval_tfms)
train_ds = Subset(full_train_ds, train_idx)
val_ds = Subset(full_eval_ds, val_idx)
test_ds = BrainMRIDataset(TEST_DIR, transform=eval_tfms)

# DataLoaders
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_loader))
print("Val batches:", len(val_loader))
print("Test batches:", len(test_loader))

Train batches: 72
Val batches: 18
Test batches: 13


In [5]:
# Construction du modèle EfficientNet-B0
# pretrained=True : on utilise un modèle déjà entraîné sur ImageNet
# C'est le Transfer Learning - le modèle connaît déjà les formes de base
model = build_efficientnet_b0(num_classes=2, pretrained=True)
model = model.to(device)
print("Modèle EfficientNet-B0 chargé ✅")
print("Device:", device)

Modèle EfficientNet-B0 chargé ✅
Device: cpu


In [6]:
# Définition de la fonction de perte et de l'optimiseur
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

print("Début de l'entraînement EfficientNet-B0 🚀")
print("-" * 50)

for epoch in range(EPOCHS):
    # Phase entraînement - retourne un dictionnaire
    train_metrics = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    # Phase validation - retourne un dictionnaire
    val_metrics = validate_one_epoch(
        model, val_loader, criterion, device
    )
    # On extrait les valeurs du dictionnaire avec ["loss"] et ["acc"]
    print(f"Epoch {epoch+1}/{EPOCHS} | "
          f"Train Loss: {train_metrics['loss']:.4f} | "
          f"Train Acc: {train_metrics['acc']:.4f} | "
          f"Val Loss: {val_metrics['loss']:.4f} | "
          f"Val Acc: {val_metrics['acc']:.4f}")

Début de l'entraînement EfficientNet-B0 🚀
--------------------------------------------------


Epoch 1/10 | Train Loss: 0.1238 | Train Acc: 0.9490 | Val Loss: 0.6638 | Val Acc: 0.6254


Epoch 2/10 | Train Loss: 0.0873 | Train Acc: 0.9678 | Val Loss: 0.1154 | Val Acc: 0.9477


Epoch 3/10 | Train Loss: 0.0454 | Train Acc: 0.9852 | Val Loss: 0.0428 | Val Acc: 0.9861


Epoch 4/10 | Train Loss: 0.0384 | Train Acc: 0.9874 | Val Loss: 0.1931 | Val Acc: 0.9686


...

Epoch 6/10 | Train Loss: 0.0285 | Train Acc: 0.9891 | Val Loss: 0.0207 | Val Acc: 0.9913


Epoch 7/10 | Train Loss: 0.0423 | Train Acc: 0.9874 | Val Loss: 0.0480 | Val Acc: 0.9861


Epoch 8/10 | Train Loss: 0.0223 | Train Acc: 0.9935 | Val Loss: 0.0432 | Val Acc: 0.9913


Epoch 9/10 | Train Loss: 0.0287 | Train Acc: 0.9904 | Val Loss: 0.0953 | Val Acc: 0.9547


Epoch 10/10 | Train Loss: 0.0164 | Train Acc: 0.9948 | Val Loss: 0.0071 | Val Acc: 0.9983


In [7]:
# Sauvegarde du modèle entraîné
import os
os.makedirs("../models", exist_ok=True)

save_checkpoint(model, "../models/efficientnet_b0_final.pt")
print("Modèle sauvegardé ✅")

Modèle sauvegardé ✅


In [8]:
# Evaluation finale sur le jeu de test
# Ce sont des images que le modèle n'a JAMAIS vues
print("Evaluation sur le jeu de test final...")
print("-" * 50)

test_metrics = validate_one_epoch(
    model, test_loader, criterion, device
)

print(f"Test Accuracy : {test_metrics['acc']:.4f}")
print(f"Test Loss     : {test_metrics['loss']:.4f}")

# Métriques détaillées
metrics = compute_metrics(test_metrics['targets'], test_metrics['preds'])
print(f"F1-Score      : {metrics['f1']:.4f}")
print(f"Recall tumor  : {metrics['recall']:.4f}")
print(f"Precision     : {metrics['precision']:.4f}")

Evaluation sur le jeu de test final...
--------------------------------------------------


Test Accuracy : 0.8452
Test Loss     : 0.4676
F1-Score      : 0.8820
Recall tumor  : 0.7889
Precision     : 1.0000


In [9]:
# Comparaison EfficientNet-B0 vs ResNet18
# Résultats de ResNet18 obtenus dans le notebook 04
print("=" * 55)
print("   COMPARAISON DES MODÈLES")
print("=" * 55)

resultats = {
    "ResNet18": {
        "accuracy": 0.9721,
        "f1": 0.9839,
        "recall": 0.9859,
        "precision": None
    },
    "EfficientNet-B0": {
        "accuracy": test_metrics['acc'],
        "f1": metrics['f1'],
        "recall": metrics['recall'],
        "precision": metrics['precision']
    }
}

print(f"{'Métrique':<20} {'ResNet18':>15} {'EfficientNet-B0':>15}")
print("-" * 55)
print(f"{'Accuracy':<20} {resultats['ResNet18']['accuracy']:>15.4f} {resultats['EfficientNet-B0']['accuracy']:>15.4f}")
print(f"{'F1-Score':<20} {resultats['ResNet18']['f1']:>15.4f} {resultats['EfficientNet-B0']['f1']:>15.4f}")
print(f"{'Recall tumor':<20} {resultats['ResNet18']['recall']:>15.4f} {resultats['EfficientNet-B0']['recall']:>15.4f}")
print("-" * 55)
print("\nConclusion :")
print("ResNet18 reste plus robuste sur le test final.")
print("EfficientNet-B0 nécessite plus de tuning (époques, sampler).")

   COMPARAISON DES MODÈLES
Métrique                    ResNet18 EfficientNet-B0
-------------------------------------------------------
Accuracy                      0.9721          0.8452
F1-Score                      0.9839          0.8820
Recall tumor                  0.9859          0.7889
-------------------------------------------------------

Conclusion :
ResNet18 reste plus robuste sur le test final.
EfficientNet-B0 nécessite plus de tuning (époques, sampler).
